# ДЗ 4. Построение RAG-системы

---

### Зачем это нужно

Промптинг позволяет изменить поведение большой языковой модели, но не ее знания. Фактическая информация хранится в параметрах модели, поэтому она не может надежно отвечать на вопросы о событиях после окончания обучения, использовать закрытые документы организации или точно воспроизводить редкие факты, даты, номера и другие специализированные сведения. В подобных ситуациях языковые модели нередко генерируют галлюцинации — правдоподобные по форме, но фактически неверные ответы.

Один из наиболее эффективных способов преодолеть это ограничение — Retrieval-Augmented Generation (RAG). Вместо изменения параметров модели система сначала находит релевантные документы во внешней коллекции, а затем передает их в контекст языковой модели, которая формирует ответ, опираясь на найденные источники.

В этом домашнем задании вы самостоятельно построите простую RAG-систему: подготовите коллекцию документов, реализуете поиск, сформируете промпт с найденными фрагментами и получите ответы языковой модели. Особое внимание будет уделено оценке качества системы. Вы отдельно измерите качество поиска и качество генерации, а также оцените устойчивость системы к галлюцинациям, в том числе на вопросах, для которых в коллекции документов отсутствует ответ.

### Что вы сделаете

1. Построите векторный индекс по корпусу документов и реализуете компонент поиска (retriever) (блок 1).
2. Оцените качество поиска с помощью метрик Recall@k и MRR (блок 2).
3. Соберете RAG-систему: выполните поиск релевантных фрагментов, сформируете промпт с опорой на найденный контекст (grounding) и ссылками на источники, после чего сгенерируете ответы языковой модели (блок 3).
4. Спроектируете и реализуете протокол оценки качества генерации, включающий три подхода: автоматические метрики, LLM-as-a-Judge метрики и ручную экспертную проверку, а также проанализируете достоинства и ограничения каждого из них (блок 4).
5. Оцените устойчивость системы к галлюцинациям на вопросах, ответы на которые отсутствуют в корпусе документов, и реализуете не менее двух способов снижения числа галлюцинаций (блок 5).
6. Проанализируете полученные результаты и сформулируете обоснованные выводы (блок 6).

> **Сколько займет:** ориентировочно 5—6 часов.

> **Про объем кода:** не превращайте ноутбук в полотно, пишите функциями, комментируйте код. Лаконичное и читаемое решение оценивается выше длинного и запутанного.

> **Требуется GPU.** В Google Colab выберите Runtime → Change runtime type → GPU.

## Настройка окружения

In [ ]:
!pip install -q -U transformers accelerate sentence-transformers faiss-cpu rank_bm25

In [ ]:
import re, json, time
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import torch

from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

EMB_MODEL = "intfloat/multilingual-e5-small" # маленькая мультиязычная модель эмбеддингов
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct" # маленькая instruct LLM
JUDGE_MODEL = "unsloth/Llama-3.2-3B-Instruct" # модель-судья для оценки качества генерации

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("GPU не найден, рекомендуется включить GPU.")

## 1. Корпус документов, векторный индекс и поиск

В этом задании вы будете работать с небольшим корпусом документов вымышленного интернет-магазина «Полка», который играет роль базы знаний. Корпус уже разбит на отдельные фрагменты (chunks). Для каждого фрагмента указаны его текст и метаданные: идентификатор (`id`), документ-источник и другая служебная информация. Корпус доступен в файле [corpus.json](https://drive.google.com/file/d/16lLsxLnVpWfM-QHlGF8y2mI3CihODXgV/).

Также вам предоставлен проверочный набор [validation_set.json](https://drive.google.com/file/d/1N8Mn0Ls1Zr9oTMbR-NJS0vHH7ckoMx7t/). Для каждого вопроса в нем известны:

- эталонный ответ;
- идентификаторы релевантных фрагментов (`gold`);
- признак `answerable`, показывающий, содержится ли ответ в корпусе документов.

Часть вопросов намеренно не имеет ответа в коллекции. Они понадобятся позднее для оценки устойчивости системы к галлюцинациям.

Все данные уже подготовлены. Скачайте файлы `corpus.json` и `validation_set.json` по ссылкам и загрузите их в рабочую директорию Google Colab. В дальнейшем предполагается, что файлы будут доступны по путям `corpus.json` и `validation_set.json`.

Ваша задача здесь — вычислить эмбеддинги для всех фрагментов корпуса, построить векторный индекс и реализовать функцию поиска `retrieve`, которая по запросу пользователя возвращает наиболее релевантные фрагменты.


In [ ]:
with open("corpus.json", "r", encoding="utf-8") as f:
    CORPUS = json.load(f)

with open("validation_set.json", "r", encoding="utf-8") as f:
    VAL = json.load(f)

print(f"Фрагментов в корпусе: {len(CORPUS)}")
print(f"Проверочных вопросов: {len(VAL)} "
      f"(отвечаемых: {sum(e["answerable"] for e in VAL)}, "
      f"out-of-corpus: {sum(not e["answerable"] for e in VAL)})")

In [ ]:
# Модель эмбеддингов и функция для кодирования текстов

# Используется семейство моделей E5, обученное различать два типа входных текстов:
# 1) запросы — с префиксом "query:"
# 2) документы — с префиксом "passage:"

# Во время обучения модель всегда видела именно такие префиксы. Если перепутать их
# или не добавить вовсе, качество семантического поиска может заметно снизиться.
# Чтобы избежать этой распространенной ошибки, используйте готовую функцию embed().

emb_model = SentenceTransformer(EMB_MODEL, device=device)

def embed(texts, kind="passage"):
    assert kind in ("passage", "query")
    prefixed = [f"{kind}: {t}" for t in texts]
    return emb_model.encode(prefixed, normalize_embeddings=True, convert_to_numpy=True)

### ✍️ Задание 1.1. Векторный индекс и поиск

1. Вычислите эмбеддинги для всех фрагментов корпуса (`kind="passage"`) и сохраните их в формате `float32`.
2. Постройте векторный индекс. Поскольку эмбеддинги уже L2-нормированы, для поиска по косинусной близости можно использовать `faiss.IndexFlatIP` (для данного корпуса также допускается реализация поиска средствами `numpy` или `sklearn`, однако функция должна возвращать оценки близости и идентификаторы найденных фрагментов.)
3. Реализуйте функцию `retrieve(query, k=4)`, которая:
    - кодирует запрос (`kind="query"`);
    - находит top-k наиболее релевантных фрагментов;
    - возвращает список пар `(фрагмент, score)`, где `фрагмент` — объект из `CORPUS`, а `score` — оценка близости.
4. Проверьте работу функции на 1—2 запросах и убедитесь, что среди первых результатов поиска находятся релевантные фрагменты.


In [ ]:
# ✍️ ВАШ КОД (задание 1.1)

## 2. Оценка качества поиска

На этом этапе вы будете оценивать только качество поиска, не учитывая качество ответов языковой модели. Это важный принцип разработки и диагностики RAG-систем: если поисковый компонент не нашел релевантный фрагмент, генеративная модель уже не сможет использовать содержащуюся в нем информацию. Поэтому поиск необходимо оценивать независимо от генерации.

Метрики поиска вычисляйте только для вопросов, на которые в корпусе существует ответ (answerable=True). Для вопросов категории out-of-corpus релевантных документов не существует, поэтому метрики поиска (Recall@k, MRR и др.) для них не определены.

### ✍️ Задание 2.1. Оцените качество поиска

Для всех вопросов с `answerable=True`:

1. Вычислите Recall@k для `k ∈ {1, 3, 5}`. В рамках этого задания будем считать вопрос успешно найденным, если хотя бы один релевантный фрагмент (`gold`) попал в первые `k` результатов поиска. Рассчитайте долю таких вопросов.
2. Вычислите MRR@10 — среднюю величину `1/rank`, где `rank` — позиция первого релевантного (`gold`) фрагмента среди первых десяти результатов поиска.
3. Выведите полученные значения и кратко прокомментируйте их.


In [ ]:
# ✍️ ВАШ КОД (задание 2.1)

### 💬 Задание 2.2. Сделайте выводы (впишите ответ ниже)

Используя полученные результаты, кратко ответьте на вопросы.

1. Как изменяется Recall@k при увеличении `k`? Какое значение `k` вы считаете разумным для данной задачи и почему?
2. Какие недостатки могут возникнуть при слишком большом значении `k`, несмотря на рост Recall@k?

> *Ваш ответ:*
>
> 1. ...
> 2. ...

## 3. Генерация ответов с помощью RAG

Итак, вы реализовали компонент поиска. Теперь необходимо объединить его с языковой моделью и построить полноценную RAG-систему.

Функция `generate(system, user)` уже реализована. Она автоматически формирует сообщения в формате чата, выполняет детерминированную генерацию (`do_sample=False`, чтобы результаты были воспроизводимыми) и возвращает только текст ответа модели.

Ваша задача — сформировать промпт, передав модели найденные фрагменты в качестве контекста (grounding), и реализовать RAG-пайплайн, который отвечает на вопросы, **опираясь только на найденные документы** и **ссылаясь на использованные источники**.

In [ ]:
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16, device_map="auto")

@torch.no_grad()
def generate(system, user, max_new_tokens=256):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    inputs = llm_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt",
    ).to(llm.device)
    outputs = llm.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=llm_tokenizer.eos_token_id,
    )
    generated = outputs[0, inputs["input_ids"].shape[1]:]
    return llm_tokenizer.decode(generated, skip_special_tokens=True).strip()

print(generate("Ты краткий ассистент.", "Скажи одно слово: работает ли генерация?"))

### ✍️ Задание 3.1. Реализуйте RAG-пайплайн с grounding и цитированием источников

1. Напишите `SYSTEM_PROMPT`, который требует от модели:
    - отвечать только на основе переданных фрагментов (grounding), не используя внешние знания;
    - сопровождать каждое фактическое утверждение ссылкой на соответствующий фрагмент в формате `[n]`;
    - если среди переданных фрагментов нет ответа на вопрос, вернуть ровно фразу `В предоставленных документах нет информации.` и не дополнять ее никакими пояснениями.
2. Реализуйте функцию `build_context(chunks)`, которая формирует контекст для модели, пронумеровав переданные фрагменты как `[1]`, `[2]`, ... в порядке их следования.
3. Реализуйте функцию `rag_answer(query, k=4)`, которая последовательно выполняет:
   ```
   retrieve → build_context → generate
   ```

   Функция должна возвращать пару:
   ```
   (answer, retrieved_chunks)
   ```
4. Проверьте работу системы на трех вопросах, для которых ответ содержится в корпусе (`answerable=True`), и одном вопросе категории out-of-corpus. Для каждого вопроса выведите ответ модели и найденные фрагменты.

> **Важно.** Ссылки вида `[n]` относятся к порядку фрагментов в сформированном контексте, а не к их исходным идентификаторам. Например, ссылка `[2]` означает второй переданный модели фрагмент (`retrieved_chunks[1]`), по которому при необходимости можно определить его исходный `id`. Такой вариант делает требования более однозначными, а пояснение про ссылки сразу объясняет, как их интерпретировать при проверке результатов.


In [ ]:
# ✍️ ВАШ КОД (задание 3.1)

## 4. Оценка качества генерации

Качество ответов RAG-системы нельзя надежно оценить с помощью одной метрики. На практике используют несколько взаимодополняющих способов оценки, каждый из которых имеет свои достоинства и ограничения.

В этом задании вы реализуете протокол оценки, состоящий из трех методов:

- **Автоматические метрики** — сравнивают ответ модели с эталонным ответом. Они вычисляются быстро и воспроизводимы, однако чувствительны к формулировкам и не позволяют оценить, действительно ли ответ основан на найденных документах.
- **LLM-as-a-Judge** — языковая модель выступает в роли эксперта и оценивает, насколько ответ соответствует найденному контексту (faithfulness) и отвечает на поставленный вопрос (answer relevance). Такой подход значительно дешевле ручной проверки, но сам подвержен ошибкам и систематическим смещениям.
- **Ручная экспертная оценка** — наиболее надежный способ проверки качества, позволяющий выявить ошибки, которые не замечают автоматические методы. Недостаток этого подхода — высокая стоимость и плохая масштабируемость.

**Важно**: в роли судьи используется отдельная языковая модель, а не та же модель, которая генерирует ответы. Желательно, чтобы модель-судья была не слабее оцениваемой модели: недостаточно сильные модели могут не распознавать ошибки, переоценивать качество ответов и пропускать очевидные нарушения. В данном задании в качестве судьи используется модель `unsloth/Llama-3.2-3B-Instruct`.

Автоматические метрики уже реализованы. Ваша задача — разработать оценку с помощью LLM-as-a-Judge, выполнить небольшую ручную проверку результатов и сравнить сильные и слабые стороны всех трех подходов.

In [ ]:
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
judge = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL, torch_dtype=torch.float16, device_map="auto")

@torch.no_grad()
def judge_generate(system, user, max_new_tokens=16):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    inputs = judge_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_tensors="pt", return_dict=True,
    ).to(judge.device)
    outputs = judge.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=judge_tokenizer.pad_token_id, eos_token_id=judge_tokenizer.eos_token_id,
    )
    prompt_len = inputs["input_ids"].shape[1]
    return judge_tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()

print(judge_generate("Отвечай ровно одним словом.", "Скажи: судья готов?"))


In [ ]:
# Строковые метрики для автоматической оценки

def normalize(s):
    if s is None:
        return ""
    s = s.lower().replace("ё", "е")
    s = re.sub(r"[^0-9a-zа-я%₽ ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def token_f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    if not p or not g:
        return 0.0
    overlap = sum((Counter(p) & Counter(g)).values())
    if overlap == 0:
        return 0.0
    prec = overlap / len(p)
    rec = overlap / len(g)
    return 2 * prec * rec / (prec + rec)

def contains_answer(pred, gold):
    # грубая проверка: нормализованный эталон входит в нормализованный ответ
    return normalize(gold) in normalize(pred)

### ✍️ Задание 4.1. Оцените качество генерации тремя способами

> **Рекомендация.** Поскольку функция `rag_answer()` работает детерминированно, сохраните сгенерированные ответы и используйте их повторно при вычислении всех метрик.

1. **Автоматические метрики.** Для всех вопросов с `answerable=True`:
    - вычислите среднее значение `token_f1`;
    - вычислите долю ответов, для которых `contains_answer=True` (ответ содержит эталонный ответ);
    - представьте результаты в виде таблицы `pandas.DataFrame`.

2. **Оценка с помощью LLM-as-a-Judge.** Реализуйте функцию `judge_faithful(question, context, answer)`, которая вызовом `judge_generate()` просит модель определить, подтверждаются ли все фактические утверждения ответа переданными фрагментами. Судья должен возвращать только один из двух вариантов: «ДА» или «НЕТ». Используя эту функцию, вычислите долю faithful-ответов среди вопросов с `answerable=True`. Дополнительно (необязательно): реализуйте аналогичную проверку для метрики answer relevance.

3. **Ручная экспертная оценка.** Выведите 3—4 пары «вопрос-ответ» и в текстовой ячейке 4.2 вручную оцените, является ли каждый ответ фактически корректным и подтверждается ли он найденными документами.

In [ ]:
# ✍️ ВАШ КОД (задание 4.1)

### 💬 Задание 4.2. Проанализируйте результаты оценки

Опираясь на результаты, полученные в задании 4.1, ответьте на следующие вопросы.

1. **Ручная проверка.** Соответствуют ли ответы модели найденным документам? Были ли среди них фактически неверные ответы или галлюцинации? Приведите 2—3 предложения с выводами по проверенным примерам.
2. **Сравнение методов оценки.** Назовите минимум по одному ограничению каждого из трех методов: автоматических метрик, LLM-as-a-Judge, ручной экспертной проверки. Если результаты этих методов различались на ваших данных, приведите один такой пример и объясните возможную причину.
3. Почему при оценке RAG-системы недостаточно использовать только один из этих методов? Как они дополняют друг друга?

> *Ваш ответ:*
>
> 1. ...
> 2. ...
> 3. ...


## 5. Оценка и снижение галлюцинаций

Даже при использовании RAG языковая модель может уверенно отвечать на вопросы, для которых в коллекции документов отсутствует ответ. Такое поведение особенно опасно, поскольку внешне ответ выглядит правдоподобным, но не имеет фактической опоры.

Для оценки этого свойства в проверочный набор специально включены вопросы категории out-of-corpus, не имеющие ответа в корпусе документов.

В этом блоке вы измерите, насколько часто система генерирует необоснованные ответы на такие вопросы, а затем реализуете не менее двух способов снижения числа галлюцинаций, основанных на разных механизмах. При этом важно оценить возникающий компромисс: уменьшается ли число галлюцинаций ценой того, что система начинает чаще отказываться отвечать даже в случаях, когда необходимая информация присутствует в корпусе.

### ✍️ Задание 5.1. Измерьте базовый уровень галлюцинаций

1. Реализуйте функцию `is_abstained(answer)`, которая определяет, отказалась ли модель отвечать. Например, функция может проверять наличие в ответе фраз вроде «нет информации», «не могу ответить» и других аналогичных формулировок.
2. Реализуйте функцию `hallucination_report(answer_fn)`, которая вычисляет для заданного RAG-пайплайна следующие показатели:
    - **abstain@ooc** — долю вопросов категории out-of-corpus, для которых модель корректно отказалась от ответа (чем выше, тем лучше);
    - **hallucination@ooc** = `1 − abstain@ooc` — долю вопросов out-of-corpus, на которые модель все же попыталась ответить (чем ниже, тем лучше);
    - **over-abstain@answerable** — долю вопросов с `answerable=True`, на которые модель ошибочно отказалась отвечать (чем ниже, тем лучше).
3. Используя эту функцию, измерьте бейзлайновый уровень галлюцинаций для реализации `rag_answer`.

> Перед выполнением задания 5.2 постройте распределения оценок близости (top-1 similarity score) отдельно для вопросов с `answerable=True` и вопросов категории out-of-corpus. Используйте полученные распределения при выборе порога: он должен опираться на наблюдаемые данные, а не выбираться произвольно.


In [ ]:
# ✍️ ВАШ КОД (задание 5.1)

### ✍️ Задание 5.2. Реализуйте не менее двух способов снижения галлюцинаций

Реализуйте не менее двух подходов, основанных на разных механизмах, и для каждого оцените качество с помощью `hallucination_report`.

В качестве примеров можно использовать следующие стратегии.

1. Подход A (изменение генерации). Усильте `SYSTEM_PROMPT`: явно запретите модели использовать знания вне переданного контекста, потребуйте воздерживаться от ответа при отсутствии информации и при желании добавьте один или несколько few-shot примеров корректного отказа.
2. Подход B (гейтинг поиска). Если оценка близости лучшего найденного фрагмента (top-1 similarity score) ниже порога `THRESH`, не вызывайте генератор, а сразу возвращайте сообщение об отсутствии информации. Значение порога выберите, опираясь на распределения, полученные в задании 5.1.

При желании реализуйте дополнительные способы (например, комбинацию нескольких подходов).

Для каждого варианта (бейзлайновый пайплайн, подход A, подход B и, при наличии, их комбинация) вычислите:

- abstain@ooc;
- hallucination@ooc;
- over-abstain@answerable.

Представьте результаты в виде сравнительной таблицы и кратко прокомментируйте, какие изменения произошли и какой компромисс между снижением галлюцинаций и избыточными отказами вы наблюдаете.

In [ ]:
# ✍️ ВАШ КОД (задание 5.2)

### 💬 Задание 5.3. Проанализируйте результаты экспериментов (впишите ответ ниже)

Используя результаты задания 5.2, ответьте на следующие вопросы.

1. Каков был бейзлайновый уровень галлюцинаций (`hallucination@ooc`)? Как изменились все три метрики после применения каждого из реализованных подходов? Приведите полученные значения.
2. Какой компромисс между снижением галлюцинаций и избыточными отказами вы наблюдали? Как изменился показатель `over-abstain@answerable`? Какой из реализованных подходов показался вам наиболее удачным и почему?
3. Если вы использовали гейтинг по оценке близости, объясните, как был выбран порог `THRESH`. Что произойдет, если сделать его значительно ниже или значительно выше?

> *Ваш ответ:*
>
> 1. ...
> 2. ...
> 3. ...

## 6. Вывод (2 балла)

### 💬 Задание 6.1. Вывод (впишите ответ ниже)

Напишите 6—9 предложений, опираясь на результаты, полученные в ходе выполнения работы. Используйте конкретные значения метрик, где это уместно.

В выводе обязательно ответьте на следующие вопросы:

1. Каким оказалось качество поиска (Recall@k, MRR) и генерации (автоматические метрики, LLM-as-a-Judge)?
2. Какой этап RAG-пайплайна оказался наиболее проблемным в вашем эксперименте? Чем это подтверждается?
3. Насколько система была склонна к галлюцинациям? Как изменились показатели после применения методов их снижения?
4. Для каких задач такую систему уже можно использовать? Какого компонента или улучшения ей не хватает до промышленного уровня? Предложите одно конкретное направление дальнейшего улучшения.

> *Ваш ответ:* ...

## Бонусное задание

> Бонусная часть не является обязательной к выполнению и не влияет на оценку за домашнее задание. Выполнив это задание, вы получите развернутую обратную связь в свободной форме. Здесь нет строгих критериев выполнения.

Плотный поиск хорошо находит семантически близкие фрагменты, но может пропускать документы с точными идентификаторами, артикулами, кодами или редкими терминами. Лексический поиск (BM25) обладает противоположными свойствами: он отлично работает на точных совпадениях, но плохо справляется с перефразированиями.

В промышленных RAG-системах эти подходы обычно объединяют с помощью гибридного поиска. Один из самых простых и эффективных способов объединения результатов - Reciprocal Rank Fusion (RRF), который учитывает только позиции документов в ранжированных списках и не требует приводить оценки разных поисковиков к общей шкале.


### ✍️ Задание Б1. Реализуйте гибридный поиск с помощью RRF

1. Постройте BM25-индекс по корпусу (например, с помощью библиотеки `rank_bm25`) и реализуйте функцию `retrieve_bm25(query, k)`, возвращающую top-k наиболее релевантных фрагментов.
2. Реализуйте функцию `retrieve_rrf(query, k)`, которая:
    - получает top-10 результатов плотного поиска и top-10 результатов BM25;
    - объединяет их с помощью Reciprocal Rank Fusion (RRF).
3. Для вопросов с `answerable=True` сравните качество:
    - плотного поиска;
    - гибридного поиска.

   Вычислите Recall@1, Recall@3 и Recall@5 для обоих вариантов и представьте результаты в виде таблицы.
4. Кратко прокомментируйте результаты. Улучшил ли гибридный поиск качество? На каких запросах он оказался полезнее плотного поиска?


In [ ]:
# ✍️ ВАШ КОД (задание Б1)